In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.diagnostics import run_c3_gmm_to_kmeans
from src.plotting import (
    plot_metric_vs_sigma2,
    plot_dual_metric_vs_sigma2,
)

In [ ]:
result = run_c3_gmm_to_kmeans(
    pca_dim=10,
    sigma2_values=(2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01),
    random_state=42,
    gmm_n_init=10,
    save_dir="../results/tables/c3_gmm_to_kmeans",
)

In [ ]:
results_df = result["results_df"]
results_df

In [ ]:
plot_metric_vs_sigma2(
    results_df,
    metric="label_agreement_test",
    title="C3: Test label agreement between GMM and KMeans",
)

plot_metric_vs_sigma2(
    results_df,
    metric="center_distance",
    title="C3: Center distance between GMM and KMeans",
)

plot_metric_vs_sigma2(
    results_df,
    metric="gmm_test_entropy",
    title="C3: Mean test entropy of GMM responsibilities",
)

In [ ]:
plot_dual_metric_vs_sigma2(
    results_df,
    metric1="label_agreement_test",
    metric2="gmm_test_entropy",
    title="C3: Agreement rises while entropy falls",
    label1="Label agreement (test)",
    label2="Mean entropy (test)",
)

plot_dual_metric_vs_sigma2(
    results_df,
    metric1="center_distance",
    metric2="gmm_test_entropy",
    title="C3: Center distance vs entropy",
    label1="Center distance",
    label2="Mean entropy (test)",
)

In [ ]:
sigma2 = 0.02
artifact = result["artifacts"][sigma2]

artifact["gmm"].weights_
artifact["gmm"].means_.shape
artifact["gmm"].log_likelihood_history_[-10:]

In [ ]:
import os
from pathlib import Path
import pandas as pd

from src.plotting import (
    plot_metric_vs_sigma2,
    plot_dual_metric_vs_sigma2,
)

# ---------- 结果目录 ----------
fig_dir = Path("../results/figures/c3_gmm_to_kmeans")
tab_dir = Path("../results/tables/c3_gmm_to_kmeans")
fig_dir.mkdir(parents=True, exist_ok=True)
tab_dir.mkdir(parents=True, exist_ok=True)

# ---------- 如果你前面还没有运行 result，这里先运行 ----------
# result = run_c3_gmm_to_kmeans(
#     pca_dim=10,
#     sigma2_values=(2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01),
#     random_state=42,
#     gmm_n_init=10,
#     save_dir=str(tab_dir),
# )

results_df = result["results_df"].copy()
results_df.to_csv(tab_dir / "c3_results_full.csv", index=False)

# ---------- 图1：label agreement ----------
plot_metric_vs_sigma2(
    results_df,
    metric="label_agreement_test",
    title="C3 Test Label Agreement between GMM and KMeans",
    save_path=str(fig_dir / "c3_label_agreement_test.png"),
)

# ---------- 图2：center distance ----------
plot_metric_vs_sigma2(
    results_df,
    metric="center_distance",
    title="C3 Center Distance between GMM and KMeans",
    save_path=str(fig_dir / "c3_center_distance.png"),
)

# ---------- 图3：test entropy ----------
plot_metric_vs_sigma2(
    results_df,
    metric="gmm_test_entropy",
    title="C3 Mean Test Entropy of GMM Responsibilities",
    save_path=str(fig_dir / "c3_test_entropy.png"),
)

# ---------- 图4：train entropy ----------
plot_metric_vs_sigma2(
    results_df,
    metric="gmm_train_entropy",
    title="C3 Mean Train Entropy of GMM Responsibilities",
    save_path=str(fig_dir / "c3_train_entropy.png"),
)

# ---------- 图5：test acc ----------
plot_metric_vs_sigma2(
    results_df,
    metric="gmm_test_acc",
    title="C3 GMM Test ACC under Different sigma^2",
    save_path=str(fig_dir / "c3_gmm_test_acc.png"),
)

# ---------- 图6：test ARI ----------
plot_metric_vs_sigma2(
    results_df,
    metric="gmm_test_ari",
    title="C3 GMM Test ARI under Different sigma^2",
    save_path=str(fig_dir / "c3_gmm_test_ari.png"),
)

# ---------- 图7：agreement vs entropy ----------
plot_dual_metric_vs_sigma2(
    results_df,
    metric1="label_agreement_test",
    metric2="gmm_test_entropy",
    title="C3 Agreement Rises while Entropy Falls",
    label1="Label agreement (test)",
    label2="Mean entropy (test)",
    save_path=str(fig_dir / "c3_agreement_vs_entropy.png"),
)

# ---------- 图8：center distance vs entropy ----------
plot_dual_metric_vs_sigma2(
    results_df,
    metric1="center_distance",
    metric2="gmm_test_entropy",
    title="C3 Center Distance vs Entropy",
    label1="Center distance",
    label2="Mean entropy (test)",
    save_path=str(fig_dir / "c3_center_distance_vs_entropy.png"),
)

# ---------- 结果摘要表 ----------
summary_cols = [
    "sigma2",
    "gmm_test_acc",
    "gmm_test_ari",
    "gmm_test_nmi",
    "gmm_test_entropy",
    "label_agreement_test",
    "center_distance",
    "iterations",
    "converged",
]
results_df[summary_cols].to_csv(tab_dir / "c3_results_summary.csv", index=False)

print("Saved figures to:", fig_dir.resolve())
print("Saved tables  to:", tab_dir.resolve())
print("\nSaved figure files:")
for p in sorted(fig_dir.glob("*.png")):
    print(" -", p.name)

print("\nSaved table files:")
for p in sorted(tab_dir.glob("*.csv")):
    print(" -", p.name)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = results_df.sort_values("sigma2")

plt.figure(figsize=(7.5, 5.5))
plt.plot(df["sigma2"], df["label_agreement_test"], marker="o", label="Label agreement")
plt.plot(df["sigma2"], df["gmm_test_entropy"], marker="s", label="Mean entropy")
plt.plot(df["sigma2"], df["gmm_test_acc"], marker="^", label="GMM test ACC")
plt.xscale("log")
plt.xlabel("Fixed shared spherical variance sigma^2")
plt.ylabel("Value")
plt.title("C3 Key Trends as sigma^2 Decreases")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "c3_key_trends.png", dpi=200, bbox_inches="tight")
plt.show()